## Version 4 – Stregkodemapper i scriptrod; input jpg/; ALTO/TXT pr. stregkode; venter ved netværksfejl og genoptager samme jpg

In [ ]:
import base64
import json
import logging
import time
from datetime import datetime
from pathlib import Path

import requests
from bs4 import BeautifulSoup


USERNAME = 'XXX'
PASSWORD = 'XXX'



def _script_base_dir():
    try:
        return Path(__vsc_ipynb_file__).resolve().parent
    except NameError:
        pass
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()


base_dir = _script_base_dir()
run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = base_dir / f'transkribus_batch_{run_timestamp}.log'
state_path = base_dir / 'transkribus_batch_state.json'

logger = logging.getLogger('transkribus_batch_v24')
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(log_path, encoding='utf-8')
console_handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(console_handler)


def now_iso():
    return datetime.now().isoformat(timespec='seconds')


def load_state():
    if not state_path.exists() or state_path.stat().st_size == 0:
        return {
            'version': '2.4',
            'run_started_at': now_iso(),
            'last_update': now_iso(),
            'status': 'INIT',
            'resume_from_next': None,
            'current_file': None,
            'last_completed': None,
            'last_failed': None,
            'counts': {
                'processed': 0,
                'success': 0,
                'failed': 0,
                'skipped': 0,
                'resumed_skip': 0
            }
        }

    try:
        with open(state_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if 'counts' not in data:
            data['counts'] = {'processed': 0, 'success': 0, 'failed': 0, 'skipped': 0, 'resumed_skip': 0}
        return data
    except Exception:
        backup_path = state_path.with_suffix('.json.broken')
        state_path.replace(backup_path)
        logger.warning(f'Kunne ikke læse state-fil. Backup gemt som: {backup_path}')
        return {
            'version': '2.4',
            'run_started_at': now_iso(),
            'last_update': now_iso(),
            'status': 'INIT',
            'resume_from_next': None,
            'current_file': None,
            'last_completed': None,
            'last_failed': None,
            'counts': {
                'processed': 0,
                'success': 0,
                'failed': 0,
                'skipped': 0,
                'resumed_skip': 0
            }
        }


def save_state(state):
    state['last_update'] = now_iso()
    with open(state_path, 'w', encoding='utf-8') as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


def file_key(barcode, image_path):
    return f'{barcode}/{image_path.name}'


def ensure_token():
    global TOKEN

    if 'TOKEN' in globals() and TOKEN:
        return TOKEN

    token_resp = requests.post(
        'https://account.readcoop.eu/auth/realms/readcoop/protocol/openid-connect/token',
        data={
            'grant_type': 'password',
            'username': USERNAME,
            'password': PASSWORD,
            'client_id': 'processing-api-client'
        }
    )

    if token_resp.status_code != 200:
        raise RuntimeError(
            f'Kunne ikke hente TOKEN | status_code={token_resp.status_code} | svar={token_resp.text[:500]}'
        )

    token_json = token_resp.json()
    TOKEN = token_json.get('access_token')
    if not TOKEN:
        raise RuntimeError(f'Mangler access_token i login-svar: {token_resp.text[:500]}')

    logger.info('TOKEN hentet automatisk')
    return TOKEN


def auth_headers():
    return {'Authorization': f'Bearer {ensure_token()}'}


REQUEST_TIMEOUT = (10, 120)
NETWORK_RETRY_WAIT = 60


def wait_for_transkribus(barcode, image_name, process_id=None, where=''):
    state['status'] = 'WAITING_TRANSCRIBUS'
    state['current_process_id'] = process_id
    save_state(state)
    extra = f' | process_id={process_id}' if process_id else ''
    place = f' | trin={where}' if where else ''
    logger.warning(
        f'Transkribus utilgængelig, venter {NETWORK_RETRY_WAIT}s'
        f' | barcode={barcode} | fil={image_name}{extra}{place}'
    )
    time.sleep(NETWORK_RETRY_WAIT)


if not base_dir.is_dir():
    raise FileNotFoundError(f'Mappen findes ikke: {base_dir}')

state = load_state()
state['status'] = 'RUNNING'
state['log_path'] = str(log_path)
save_state(state)

logger.info(f'Batch v2.4 start | base_dir={base_dir} | state={state_path} | log={log_path}')

processed_total = state['counts'].get('processed', 0)
success_total = state['counts'].get('success', 0)
failed_total = state['counts'].get('failed', 0)
skipped_total = state['counts'].get('skipped', 0)
resumed_total = state['counts'].get('resumed_skip', 0)

barcode_dirs = sorted(
    p
    for p in base_dir.iterdir()
    if p.is_dir() and not p.name.startswith('.') and (p / 'jpg').is_dir()
)
if not barcode_dirs:
    logger.warning(f'Ingen stregkodemapper med undermappe jpg/ fundet i: {base_dir}')

for barcode_dir in barcode_dirs:
    barcode = barcode_dir.name
    jpg_dir = barcode_dir / 'jpg'

    jpg_map = {}
    for candidate in sorted(jpg_dir.glob('*')):
        if candidate.is_file() and candidate.suffix.lower() == '.jpg':
            jpg_map[candidate.name.lower()] = candidate
    jpg_files = sorted(jpg_map.values(), key=lambda p: p.name.lower())

    if not jpg_files:
        skipped_total += 1
        state['counts']['skipped'] = skipped_total
        state['status'] = 'SKIPPED_NO_JPG'
        save_state(state)
        logger.warning(f'Springer over barcode={barcode} | Ingen jpg-filer i: {jpg_dir}')
        continue

    barcode_alto_dir = barcode_dir / 'transkribus_alto_filer'
    barcode_txt_dir = barcode_dir / 'transkribus_txt_filer'
    barcode_alto_dir.mkdir(parents=True, exist_ok=True)
    barcode_txt_dir.mkdir(parents=True, exist_ok=True)

    for image_path in jpg_files:
        current_key = file_key(barcode, image_path)
        xml_path = barcode_alto_dir / f'{image_path.stem}.xml'
        txt_path = barcode_txt_dir / f'{image_path.stem}.txt'

        xml_ready = xml_path.exists() and xml_path.stat().st_size > 0
        txt_ready = txt_path.exists() and txt_path.stat().st_size > 0

        if xml_ready and txt_ready:
            resumed_total += 1
            state['counts']['resumed_skip'] = resumed_total
            state['current_file'] = current_key
            state['status'] = 'RESUME_SKIP_ALREADY_DONE'
            save_state(state)
            logger.info(f'Genoptagelse: springer færdig fil over | barcode={barcode} | fil={image_path.name}')
            continue



        logger.info(f'Start fil | barcode={barcode} | fil={image_path.name}')

        process_id = None
        poll_finished = False
        file_done = False
        counted_this_file = False

        while not file_done:
            try:
                if not counted_this_file:
                    processed_total += 1
                    state['counts']['processed'] = processed_total
                    counted_this_file = True

                state['current_file'] = current_key
                state['resume_from_next'] = current_key
                save_state(state)

                if process_id is None:
                    state['status'] = 'PROCESSING'
                    save_state(state)

                    with open(image_path, 'rb') as f:
                        image_bytes = f.read()
                    img_base64 = base64.b64encode(image_bytes).decode()

                    while True:
                        try:
                            create_resp = requests.post(
                                'https://transkribus.eu/processing/v1/processes',
                                headers=auth_headers(),
                                json={
                                    'config': {'textRecognition': {'htrId': 356425}},
                                    'image': {'base64': img_base64}
                                },
                                timeout=REQUEST_TIMEOUT
                            )
                            break
                        except requests.exceptions.RequestException:
                            wait_for_transkribus(barcode, image_path.name, where='create')

                    if create_resp.status_code == 401:
                        TOKEN = None
                        while True:
                            try:
                                create_resp = requests.post(
                                    'https://transkribus.eu/processing/v1/processes',
                                    headers=auth_headers(),
                                    json={
                                        'config': {'textRecognition': {'htrId': 356425}},
                                        'image': {'base64': img_base64}
                                    },
                                    timeout=REQUEST_TIMEOUT
                                )
                                break
                            except requests.exceptions.RequestException:
                                wait_for_transkribus(barcode, image_path.name, where='create_retry_401')

                    if create_resp.status_code != 200:
                        failed_total += 1
                        state['counts']['failed'] = failed_total
                        state['last_failed'] = {
                            'file': current_key,
                            'reason': f'create_status_{create_resp.status_code}',
                            'snippet': create_resp.text[:300],
                            'at': now_iso()
                        }
                        state['status'] = 'FAILED_CREATE'
                        save_state(state)
                        logger.error(
                            f'Kunne ikke oprette process | barcode={barcode} | fil={image_path.name} '
                            f'| status_code={create_resp.status_code} | svar={create_resp.text[:500]}'
                        )
                        file_done = True
                        break

                    try:
                        job = create_resp.json()
                    except Exception:
                        failed_total += 1
                        state['counts']['failed'] = failed_total
                        state['last_failed'] = {
                            'file': current_key,
                            'reason': 'create_not_json',
                            'snippet': create_resp.text[:300],
                            'at': now_iso()
                        }
                        state['status'] = 'FAILED_CREATE_JSON'
                        save_state(state)
                        logger.error(f'Create-svar var ikke JSON | barcode={barcode} | fil={image_path.name}')
                        file_done = True
                        break

                    process_id = job.get('processId')
                    if not process_id:
                        failed_total += 1
                        state['counts']['failed'] = failed_total
                        state['last_failed'] = {
                            'file': current_key,
                            'reason': 'missing_process_id',
                            'snippet': json.dumps(job, ensure_ascii=False)[:300],
                            'at': now_iso()
                        }
                        state['status'] = 'FAILED_NO_PROCESS_ID'
                        save_state(state)
                        logger.error(f'Mangler processId | barcode={barcode} | fil={image_path.name}')
                        file_done = True
                        break

                    state['current_process_id'] = process_id
                    save_state(state)

                if not poll_finished:
                    state['status'] = 'POLLING'
                    save_state(state)

                    final_status = None
                    status_payload = None
                    decode_errors = 0

                    while True:
                        try:
                            status_resp = requests.get(
                                f'https://transkribus.eu/processing/v1/processes/{process_id}',
                                headers=auth_headers(),
                                timeout=REQUEST_TIMEOUT
                            )
                        except requests.exceptions.RequestException:
                            wait_for_transkribus(
                                barcode, image_path.name, process_id=process_id, where='poll'
                            )
                            continue

                        if status_resp.status_code == 401:
                            TOKEN = None
                            logger.warning(
                                f'401 ved poll, henter nyt token | barcode={barcode} | fil={image_path.name}'
                            )
                            time.sleep(1)
                            continue

                        body = status_resp.text.strip()
                        if not body:
                            decode_errors += 1
                            logger.warning(
                                f'Tomt status-svar | barcode={barcode} | fil={image_path.name} '
                                f'| process_id={process_id} | forsøg={decode_errors}'
                            )
                            if decode_errors >= 3:
                                final_status = 'FAILED'
                                status_payload = {'error': 'Gentagne tomme status-svar'}
                                break
                            time.sleep(5)
                            continue

                        try:
                            status_payload = status_resp.json()
                        except Exception:
                            decode_errors += 1
                            logger.warning(
                                f'Ugyldigt JSON i status-svar | barcode={barcode} | fil={image_path.name} '
                                f'| process_id={process_id} | forsøg={decode_errors} | svar={body[:300]}'
                            )
                            if decode_errors >= 3:
                                final_status = 'FAILED'
                                status_payload = {'error': 'Gentagne JSONDecode-fejl i status-svar'}
                                break
                            time.sleep(5)
                            continue

                        decode_errors = 0
                        current = status_payload.get('status', 'UNKNOWN')
                        logger.info(
                            f'Poll status | barcode={barcode} | fil={image_path.name} '
                            f'| process_id={process_id} | status={current}'
                        )

                        if current == 'FINISHED':
                            final_status = 'FINISHED'
                            break
                        if current == 'FAILED':
                            final_status = 'FAILED'
                            break

                        time.sleep(10)

                    if final_status == 'FAILED':
                        failed_total += 1
                        state['counts']['failed'] = failed_total
                        state['last_failed'] = {
                            'file': current_key,
                            'reason': 'poll_failed',
                            'snippet': json.dumps(status_payload, ensure_ascii=False)[:500],
                            'at': now_iso()
                        }
                        state['status'] = 'FAILED_POLL'
                        save_state(state)
                        logger.error(
                            f'Process fejlede | barcode={barcode} | fil={image_path.name} | process_id={process_id} '
                            f'| status={json.dumps(status_payload, ensure_ascii=False)}'
                        )
                        file_done = True
                        break

                    poll_finished = True

                state['status'] = 'FETCHING_ALTO'
                save_state(state)

                while True:
                    try:
                        alto_resp = requests.get(
                            f'https://transkribus.eu/processing/v1/processes/{process_id}/alto',
                            headers=auth_headers(),
                            timeout=REQUEST_TIMEOUT
                        )
                        break
                    except requests.exceptions.RequestException:
                        wait_for_transkribus(
                            barcode, image_path.name, process_id=process_id, where='alto'
                        )

                if alto_resp.status_code == 401:
                    TOKEN = None
                    while True:
                        try:
                            alto_resp = requests.get(
                                f'https://transkribus.eu/processing/v1/processes/{process_id}/alto',
                                headers=auth_headers(),
                                timeout=REQUEST_TIMEOUT
                            )
                            break
                        except requests.exceptions.RequestException:
                            wait_for_transkribus(
                                barcode, image_path.name, process_id=process_id, where='alto_retry_401'
                            )

                if alto_resp.status_code != 200:
                    failed_total += 1
                    state['counts']['failed'] = failed_total
                    state['last_failed'] = {
                        'file': current_key,
                        'reason': f'alto_status_{alto_resp.status_code}',
                        'snippet': alto_resp.text[:300],
                        'at': now_iso()
                    }
                    state['status'] = 'FAILED_ALTO'
                    save_state(state)
                    logger.error(
                        f'Kunne ikke hente ALTO | barcode={barcode} | fil={image_path.name} '
                        f'| process_id={process_id} | status_code={alto_resp.status_code} | svar={alto_resp.text[:500]}'
                    )
                    file_done = True
                    break

                alto_xml = alto_resp.text
                if not alto_xml.strip():
                    failed_total += 1
                    state['counts']['failed'] = failed_total
                    state['last_failed'] = {
                        'file': current_key,
                        'reason': 'empty_alto',
                        'snippet': '',
                        'at': now_iso()
                    }
                    state['status'] = 'FAILED_EMPTY_ALTO'
                    save_state(state)
                    logger.error(
                        f'Tomt ALTO-svar | barcode={barcode} | fil={image_path.name} | process_id={process_id}'
                    )
                    file_done = True
                    break

                with open(xml_path, 'w', encoding='utf-8') as f:
                    f.write(alto_xml)

                soup = BeautifulSoup(alto_xml, 'xml')
                lines = []
                for line in soup.find_all('TextLine'):
                    words = ' '.join(s.get('CONTENT', '') for s in line.find_all('String')).strip()
                    if words:
                        lines.append(words)

                with open(txt_path, 'w', encoding='utf-8') as f:
                    f.write(chr(10).join(lines))

                success_total += 1
                state['counts']['success'] = success_total
                state['last_completed'] = {
                    'file': current_key,
                    'xml_path': str(xml_path),
                    'txt_path': str(txt_path),
                    'at': now_iso()
                }
                state['resume_from_next'] = current_key
                state['status'] = 'DONE_FILE'
                state.pop('current_process_id', None)
                save_state(state)

                logger.info(
                    f'Færdig fil | barcode={barcode} | fil={image_path.name} '
                    f'| xml={xml_path} | txt={txt_path}'
                )
                file_done = True

            except requests.exceptions.RequestException as e:
                wait_for_transkribus(
                    barcode, image_path.name, process_id=process_id, where=f'request: {e}'
                )

            except Exception as e:
                failed_total += 1
                state['counts']['failed'] = failed_total
                state['last_failed'] = {
                    'file': current_key,
                    'reason': f'unexpected: {e}',
                    'snippet': '',
                    'at': now_iso()
                }
                state['status'] = 'FAILED_EXCEPTION'
                save_state(state)
                logger.exception(f'Uventet fejl | barcode={barcode} | fil={image_path.name} | fejl={e}')
                file_done = True


state['status'] = 'FINISHED_RUN'
state['current_file'] = None
state['summary'] = {
    'processed': processed_total,
    'success': success_total,
    'failed': failed_total,
    'skipped': skipped_total,
    'resumed_skip': resumed_total,
    'log_path': str(log_path),
    'finished_at': now_iso()
}
save_state(state)

logger.info(
    f'Batch v2.4 slut | behandlet={processed_total} | succes={success_total} '
    f'| failed={failed_total} | skipped={skipped_total} | genoptaget_skip={resumed_total} '
    f'| state={state_path} | log={log_path}'
)

print('Batch v2.4 færdig')
print(f'- Ny-behandlede billeder: {processed_total}')
print(f'- Succes: {success_total}')
print(f'- Failed: {failed_total}')
print(f'- Skipped (ingen jpg i jpg-mappe): {skipped_total}')
print(f'- Genoptagelse: overhoppede færdige filer: {resumed_total}')
print(f'- Logfil: {log_path}')
print(f'- State-fil: {state_path}')
print(f'- Stregkode-rod (script): {base_dir}')
print('- ALTO: <stregkode>/transkribus_alto_filer/')
print('- TXT: <stregkode>/transkribus_txt_filer/')
